In [2]:
import pandas as pd

csv_file= pd.read_csv('Electronic_sales_Sep2023-Sep2024.csv')

# Задание 1 (синтаксис)

# Для каждого покупателя получите таблицу со столбцами:
#  • preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
#  • total_spent - сумма всех трат;
#  • addons_spent - сумма на дополнительные услуги и аксессуары;
#  • orders_count - число заказов.
 
#  Отсортируйте по total_spent по убыванию и выведите топ-10.

# Пример вывода (формат; числа зависят от датасета):

#    customer_id  preferred_payment  total_spent  addons_spent  orders_count
#  0         C042     Credit Card              15420.50        890.00             7
#  1         C018     PayPal                     14810.00        450.00             5
#  2         C103     Debit Card               13990.25       1200.00             6
#  ...

# Пограничные случаи:

# • Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
#  • Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
#  • Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
#  • Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.


# print(csv_file.head())
#Пограничные случаи
# Два метода оплаты с одинаковой частотой - по умолчанию метод value counts сортирует элементы при одинаковых значениях в алфавитном порядке, в этом порядке я оставил выборку
# Пропуски в сумме или методе оплаты - удалил такие заказы

csv_file = csv_file.dropna(subset=['Payment Method','Total Price'])
result = csv_file.groupby('Customer ID', as_index=False).agg(
    preferred_payment=('Payment Method', lambda x: x.value_counts().index[0]),
    total_spent=('Total Price','sum'),
    addons_spent=('Add-on Total','sum'),
    orders_count=('Customer ID', 'count')
    
    
)

result.columns = [col.lower().replace(' ', '_') for col in result.columns]


result['addons_spent'] = result['addons_spent'].fillna(0)

sorted_res = result.sort_values('total_spent',ascending=False).reset_index(drop=True)


print(sorted_res.head(10))




   customer_id preferred_payment  total_spent  addons_spent  orders_count
0        16357     Bank Transfer     34563.70        495.19             7
1        16863            PayPal     33035.92        486.17             5
2        13813       Credit Card     31830.16        268.64             5
3        11476            PayPal     31077.61        301.59             5
4        12276     Bank Transfer     30961.18        329.07             6
5        13635       Credit Card     30260.36        480.73             5
6        12749       Credit Card     29394.56        619.43             5
7        15399            PayPal     29084.88        305.39             3
8        12319     Bank Transfer     27352.32        226.38             3
9        19996            PayPal     27296.78        432.12             6


In [ ]:
# Задание 2 (повышенная сложность)

# Постройте пайплайн на Pandas (без циклов по строкам):
 
#  1. Очистка: типы дат, пропуски, аномалии (отрицательные суммы / нулевые цены) - кратко опишите, что отфильтровали.
#  2. Метрики:
#     • доход по методу доставки;
#     • доход по типу продукта;
#     • доход по доп. услугам: по месяцам и по кварталам;
#     • cohort-анализ: когорта = месяц первого заказа покупателя; для каждой когорты - выручка и число покупателей в следующие 0, 1, 2, 3 месяца.
#  3. Визуализация (минимум 3 графика): bar (доставка / тип продукта), line (доп. услуги по месяцам), heatmap когортной матрицы выручки.
#  4. Выводы: несколько предложений по графикам. (?необязательно)
 
#  Дополнительно: оформите шаги 1–2 как функции clean(df) → df и build_metrics(df) → dict[str, DataFrame].

# Пример вывода метрик (фрагменты):

# >>> metrics['revenue_by_shipping']
#  shipping_method   revenue
#  Express           125400.50
#  Standard           98320.00
#  Pickup             45110.75
 
#  >>> metrics['addons_by_month'].head()
#  month      addons_revenue
#  2023-01           4200.00
#  2023-02           3890.50
#  2023-03           5100.00
 
#  >>> metrics['cohort_revenue']  # строки — когорта, столбцы — месяц жизни 0..3
#  cohort     0         1         2         3
#  2023-01  15200.0   8100.0   4300.0   2100.0
#  2023-02  14100.0   7600.0   3900.0      NaN
#  ...

# Графики: три отдельных figure/axes с подписями осей и заголовками. Числа в примере иллюстративные.

# Пограничные случаи:

# • Заказы с отрицательной или нулевой ценой - не должны попадать в доход после clean.
#  • Покупатель с одним заказом - в когортной матрице заполнен только месяц 0.
#  • Месяц без доп. услуг - в addons_by_month строка с 0 или отсутствие месяца (выберите и зафиксируйте поведение).
#  • Пустой датасет после очистки → функции не падают, возвращают пустые таблицы.